# JUMP — Reproducible phenotype validation (clean notebook)

This notebook reproduces the final **exploratory plate-level CRISPR/ORF morphological deviation statistics** from official JUMP profile files and official target annotations. Run in Kaggle with **Internet enabled** and packages `pandas`, `numpy`, `fsspec`, `pyarrow`, `matplotlib`, `scipy`, and `statsmodels`.

**Method:** Plate-matched negative controls; 60/40 reference/held-out split (seed 42); median/MAD feature normalization; mean absolute Z; plate-level positive-excess one-sided exact sign tests; BH adjustment within each modality. Results are exploratory, not independent biological or causal validation.

**Outputs:** `/kaggle/working/JUMP_FINAL/` and `/kaggle/working/JUMP_FINAL_REPRODUCIBLE_PACKAGE.zip`. The package includes summary CSVs and plots, not full profiles, the PubMed evidence audit, or the original multistage notebook. Model API keys are not needed.

**Caveat:** The same gene can appear in multiple target wells on a plate; this script averages by plate before testing. Plate independence and between-batch reproducibility have not been established. A single fixed random control split should not be described as a sensitivity analysis across splits.

In [ ]:
import pandas as pd, numpy as np, fsspec, pyarrow.parquet as pq
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
from pathlib import Path
import zipfile, json, platform

B = Path("/kaggle/working/JUMP_FINAL")
B.mkdir(exist_ok=True)
rng = np.random.default_rng(42)

root = "https://cellpainting-gallery.s3.amazonaws.com/cpg0016-jump-assembled/source_all/workspace/profiles_assembled/"
meta = "https://raw.githubusercontent.com/jump-cellpainting/datasets/main/metadata/"
genes = ["AKT1", "BRCA1", "EGFR", "MTOR", "TP53"]

config = {
    "CRISPR": (
        "CRISPR/v1.0a/profiles_wellpos_cc_var_mad_outlier_featselect_sphering_harmony_PCA_corrected.parquet",
        "source_13",
        ["JCP2022_800001", "JCP2022_800002"]
    ),
    "ORF": (
        "ORF/v1.0a/profiles_wellpos_cc_var_mad_outlier_featselect_sphering_harmony.parquet",
        "source_4",
        ["JCP2022_915128", "JCP2022_915129",
         "JCP2022_915130", "JCP2022_915131"]
    )
}

all_results = []

for mode, (file, source, neg_ids) in config.items():
    print("PROCESSING:", mode, flush=True)

    annotation = pd.read_csv(meta + mode.lower() + ".csv.gz")
    annotation = annotation[annotation.Metadata_Symbol.isin(genes)]
    gene_map = dict(zip(annotation.Metadata_JCP2022,
                        annotation.Metadata_Symbol))

    with fsspec.open(root + file, "rb") as stream:
        d = pq.read_table(stream).to_pandas()

    targets = d.Metadata_JCP2022.isin(gene_map)
    plates = set(d.loc[targets, "Metadata_Plate"])

    d = d[
        (d.Metadata_Source == source) &
        d.Metadata_Plate.isin(plates) &
        d.Metadata_JCP2022.isin(set(gene_map) | set(neg_ids))
    ].copy()

    d["gene"] = d.Metadata_JCP2022.map(gene_map)
    d["group"] = np.where(d.gene.notna(), "TARGET", "NEGATIVE")

    features = [c for c in d if c not in
                ["Metadata_Source", "Metadata_Plate",
                 "Metadata_Well", "Metadata_JCP2022",
                 "gene", "group"]]

    d[features] = d[features].apply(pd.to_numeric, errors="coerce")
    rows = []

    for plate, g in d.groupby("Metadata_Plate"):
        neg = g[g.group == "NEGATIVE"]
        tar = g[g.group == "TARGET"]
        if len(neg) < 10 or tar.empty:
            continue

        ix = rng.permutation(len(neg))
        cut = int(len(ix) * 0.6)
        ref, hold = neg.iloc[ix[:cut]], neg.iloc[ix[cut:]]

        med = ref[features].median()
        mad = (ref[features] - med).abs().median()
        scale = (1.4826 * mad).replace(0, np.nan)
        valid = scale.index[scale.notna()]

        if len(valid) < 5:
            continue

        def score(x):
            z = (x[valid] - med[valid]) / scale[valid]
            return z.abs().mean(axis=1)

        baseline = score(hold).median()

        for gene, sample in tar.groupby("gene"):
            for value in score(sample).dropna():
                rows.append({
                    "modality": mode, "gene": gene,
                    "plate": str(plate),
                    "score": float(value),
                    "excess": float(value - baseline),
                    "features": len(valid)
                })

    profiles = pd.DataFrame(rows)
    profiles.to_csv(B / f"{mode}_PROFILE_RESULTS.csv", index=False)

    for gene, g in profiles.groupby("gene"):
        by_plate = g.groupby("plate").excess.mean()
        k, n = int((by_plate > 0).sum()), len(by_plate)

        all_results.append({
            "modality": mode, "gene": gene,
            "profiles": len(g), "plates": n,
            "positive_plates": k,
            "mean_abs_z": g.score.mean(),
            "median_excess": by_plate.median(),
            "p_exact": binomtest(k, n, 0.5,
                               alternative="greater").pvalue
        })

    print("DONE:", mode, "profiles:", len(profiles))

final = pd.DataFrame(all_results)
final["q_bh"] = np.nan

for mode in final.modality.unique():
    mask = final.modality == mode
    final.loc[mask, "q_bh"] = multipletests(
        final.loc[mask, "p_exact"], method="fdr_bh"
    )[1]

final["passes_fdr_05"] = final.q_bh < 0.05
final.to_csv(B / "FINAL_PLATE_VALIDATION.csv", index=False)

for mode, g in final.groupby("modality"):
    g = g.sort_values("mean_abs_z", ascending=False)
    plt.figure(figsize=(7, 4))
    plt.bar(g.gene, g.mean_abs_z)
    plt.title(mode + " — JUMP morphological deviations")
    plt.ylabel("Mean absolute robust Z")
    plt.tight_layout()
    plt.savefig(B / f"{mode}_FIGURE.png", dpi=200)
    plt.close()

manifest = {
    "project": "JUMP DISCOVERY-AGENT",
    "seed": 42,
    "python": platform.python_version(),
    "analysis": "Full-feature plate-level exploratory sign tests",
    "limitation": "Independent biological replication unverified"
}
(B / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))

zip_path = Path("/kaggle/working/JUMP_FINAL_REPRODUCIBLE_PACKAGE.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in B.iterdir():
        if f.is_file():
            z.write(f, f.name)

print("\nFINAL RESULTS")
print(final.round(5).to_string(index=False))
print("\nZIP CREATED:", zip_path)
print("SIZE MB:", round(zip_path.stat().st_size / 1e6, 2))